# Milestone 1 lab, notebook 1 of 7: The question and the source

## Where we are

| Notebook | What you do |
|---|---|
| **1. The question and the source** (you are here) | Read the question. Meet the order system. Look at one order |
| 2. Read every order | Follow the pages of the API to the end |
| 3. The answer by hand | Work out the answer with plain Python. Meet the time zone trap |
| 4. Transform, then load | Put the orders in a warehouse table. Ask it one SQL question |
| 5. Every morning | Run it again the next day. Watch an old number change |
| 6. Why batch, and where it breaks | Measure how late the report is. Then watch the load fail |
| 7. The production version | The same logic as a tested program |

**Before you start,** open [`milestone_1_flow.html`](../milestone_1_flow.html) in a browser. It shows the whole pipeline you are about to build.

> Please run each cell and read its output before you move on.

## What this notebook is for

Arden is a fictional online grocery market. It delivers the same day in three cities.

Every morning, Arden's operations manager asks one question. Today a person answers it by hand, from a file. You are going to answer it with a program.

In this notebook you write almost no code. You read the question, and you look at where the data lives.

In [1]:
import json

from lab_helpers import api, check_api, set_morning, show_doc

`lab_helpers.py` is a small file beside these notebooks. It holds code that is not the lesson, such as showing a document. Open it any time.

## Step 1: Read the question

A real project starts with a requirement, not with code. This milestone has three short design documents in its `docs/` folder. The first is the **BRD** (business requirements document). It says what the business needs, in business words.

In [2]:
show_doc("BRD.md", "3. The question to answer")

**From `docs/BRD.md`**

> ## 3. The question to answer
> 
> > How many orders did we take yesterday, how much money did they bring in, how does that compare with the day before, and has any earlier day changed since I last looked?
> 
> It is four questions in one:
> 
> | # | Part | Example of an answer |
> |---|---|---|
> | 1 | How many orders yesterday? | 30 orders |
> | 2 | How much money? | $807.11 |
> | 3 | Against the day before? | 3 fewer orders, $20.40 more |
> | 4 | Did an earlier day change? | Friday went down by $5.78, because of one refund |

Keep those four parts in mind. Every notebook in this lab works toward them.

## Step 2: Why can't the manager just look in the app?

In [3]:
show_doc("BRD.md", "1. The business problem")

**From `docs/BRD.md`**

> ## 1. The business problem
> 
> Arden is a fictional online grocery market. Customers order in an app. Arden delivers the same day in three cities.
> 
> Every morning the operations manager needs yesterday's numbers. Today a person makes the report by hand, from a file exported from the admin page. That person is sometimes ill, sometimes late, and sometimes wrong.
> 
> Arden wants the same report made by a program, with the same steps every day.
> 
> *(There is a diagram here. Open `docs/BRD.md` to see it.)*
> 
> **Why copy the data at all?** The order system has one job: take orders fast. A heavy report slows the app down for customers. So nobody may run reports on it. The data is copied out to a second place, the warehouse.

So there are two places:

| | The order system | The warehouse |
|---|---|---|
| Its job | Take orders fast | Answer questions |
| Who uses it | The app, all day | People who read reports |
| A heavy report here | Slows the app for customers | Is what it was built for |

The data is **made** in the first place. It is **used** in the second. Something must carry it across, every day.

## Step 3: Meet the order system

The order system has an **API**. An API is a way for one program to ask another program for data.

It should already be running in a terminal. This cell checks.

In [6]:
check_api()

The order system is running. Its clock says 2026-06-02 07:00:00 in Arden.


This order system is a fake, so its clock is yours to move. For this lab, it is the morning of **Monday 8 June 2026**. The manager wants the report for the weekend.

In [7]:
set_morning("2026-06-08")

It is now Monday 2026-06-08 07:00:00 in Arden.


Now ask the API for orders. One line of Python sends the request. The answer comes back as JSON: names and values, as text.

In [8]:
answer = api.get("/orders")
body = answer.json()

print("Status code:", answer.status_code, "(200 means the call worked)")
print("The answer has these parts:", list(body))
print("Orders in this answer:", len(body["data"]))

Status code: 200 (200 means the call worked)
The answer has these parts: ['data', 'next_cursor']
Orders in this answer: 25


## Step 4: Look at one order

Here is the last order of that answer, exactly as the API sent it.

In [9]:
order = body["data"][-1]
print(json.dumps(order, indent=2))

{
  "order_id": 1027,
  "customer_id": 1104,
  "city": "NGT",
  "status": "delivered",
  "placed_at": "2026-06-02T03:53:19Z",
  "updated_at": "2026-06-02T14:48:00Z",
  "delivered_at": "2026-06-02T14:48:00Z",
  "cancelled_at": null,
  "total_cents": 3158,
  "refunded_cents": 0,
  "refunded_at": null,
  "items": [
    {
      "line_no": 1,
      "product_id": "P-004",
      "product_name": "Eggs, 12",
      "quantity": 1,
      "unit_price_cents": 489,
      "item_status": "ok"
    },
    {
      "line_no": 2,
      "product_id": "P-008",
      "product_name": "Salmon fillet",
      "quantity": 2,
      "unit_price_cents": 1195,
      "item_status": "ok"
    },
    {
      "line_no": 3,
      "product_id": "P-028",
      "product_name": "Dark chocolate 100g",
      "quantity": 1,
      "unit_price_cents": 279,
      "item_status": "ok"
    }
  ]
}


Read it slowly. Three things matter for the rest of the project:

| Field | What to notice |
|---|---|
| `status` | An order moves: `placed` → `packed` → `out_for_delivery` → `delivered`. It can also be `cancelled` |
| `placed_at`, `updated_at` | Two different times. Both end in `Z`, which means **UTC**, not Arden's clock |
| `total_cents` | Money in whole cents. `3158` means $31.58. No decimals, so no rounding mistakes |

## Step 5: Three new words

You have now seen all three things these words name.

| Word | Meaning | At Arden |
|---|---|---|
| **Source** | Where the data is made | The order system |
| **Destination** | Where the data is used | The warehouse, and the manager's report |
| **Data pipeline** | "A data pipeline moves data from where it is made to where it is used. It works in steps, and it runs on a schedule." | What you build in this project |

The definition in quotes is from the D8Loop chapter [What a Data Pipeline Is & Why It Matters](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/what-a-data-pipeline-is-and-why-it-matters). Read it for the full theory.

### Compare it with its opposite: copying by hand

| Copying by hand | What goes wrong | A pipeline |
|---|---|---|
| The person is ill | No report that day | Runs on a schedule |
| They forget a rule | Cancelled orders are counted as sales | The same written steps, every time |
| They cannot remember what they did | Nobody can repeat it | Code, and a record of every run |

One query is not a pipeline. A pipeline is the whole chain, and it runs again and again without help.

## What you now know

- The question has four parts: orders yesterday, money yesterday, the day before, and earlier days that changed.
- Nobody may run reports on the order system. The data must be copied out.
- One call to `GET /orders` gave 25 orders, not all of them. The answer also had a `next_cursor`.

**Say it in an interview:** "A data pipeline moves data from a source to a destination, in steps, on a schedule. In my project the source was an order API and the destination was a warehouse table that answered one question every morning."

Next: [notebook 2](02_read_every_order.ipynb) follows that `next_cursor` to read every order.